# Experiment tracking with a vector DB

This notebook illustrates experiment tracking for document-based models, using the LangChain API to integrate directly with vector databases. Users can track documents as artifacts, complete with metadata such as loader type, producer information and collection details.

In this tutorial:
- [Getting started](#getting-started)
- [Milvus configuration](#milvus-configuration)
- [Creating an MLRun collection from Milvus](#creating-an-mlrun-collection-from-milvus)
- [Create the Milvus vector store](#create-the-milvus-vector-store)
- [Working with LangChain Documents and MLRun Artifacts](#working-with-langchain-documents-and-mlrun-artifacts)
- [Working with MLRun artifacts in a collection](#working-with-mlrun-artifacts-in-a-collection)
- [Using Text Splitters with Artifacts](#using-text-splitters-with-artifacts)

## Getting started


In [119]:
import sys

sys.path.insert(0, "/home/iguazio/mlrun")

Setup and Imports

In [ ]:
import mlrun
import tempfile
from langchain.embeddings import FakeEmbeddings
from langchain_community.vectorstores import Milvus
from langchain.text_splitter import CharacterTextSplitter
from langchain_community.document_loaders import DirectoryLoader
from mlrun.artifacts import DocumentLoaderSpec, MLRunLoader
from mlrun.datastore.datastore_profile import (
    ConfigProfile,
    register_temporary_client_datastore_profile,
)

# Initialize project
project = mlrun.get_or_create_project("vectorstore-demo3")

# Set up the

## Milvus configuration

Create and register a profile representing a Milvus DB. This is done in the project level, and happens once per project.</br>
Credentials for the DB may be passed here, assuming the code is not introduced into any repo, or they may be provided through project secrets.</br>
This is performed once per project.

In [120]:
profile = ConfigProfile(
    name="milvus-config", public={"MILVUS_DB": {"host": "localhost", "port": 19530}}
)
# Register the profile temporarily for the current client session
register_temporary_client_datastore_profile(profile)

> 2024-12-12 12:13:45,553 [info] Project loaded successfully: {"project_name":"vectorstore-demo3"}


## Creating an MLRun collection from Milvus

Create (or use an existing) collection to store the artifact/documents in.</br>
Use the configuration stored earlier in the ConfigProfile to get the configuration details.</br>
You still need to create the actual VectorDB class, since each VectorDB has a different initialization method.

In [ ]:
# Initialize embedding model (using FakeEmbeddings for demonstration)
embedding_model = FakeEmbeddings(size=3)

config = project.get_config_profile_attributes("milvus-config")

## Create the Milvus vector store

In this step you also create the MLRun collection wrapper.

In [121]:
vectorstore = Milvus(
    collection_name="my_tutorial_collection",
    embedding_function=embedding_model,
    connection_args=config["MILVUS_DB"],
    auto_id=True,
)

# Create MLRun collection wrapper
collection = project.get_vector_store_collection(vector_store=vectorstore)

## Working with LangChain documents and MLRun artifacts

In [122]:
# Create a sample document
def create_sample_document(content, dir=None):
    with tempfile.NamedTemporaryFile(
        mode="w", suffix=".txt", delete=False, dir=dir
    ) as temp_file:
        temp_file.write(content)
        return temp_file.name


# Create and log an MLRun artifact
file_path = create_sample_document("Sample content for demonstration")
artifact = project.log_document("sample-doc", local_path=file_path)

# Convert MLRun artifact to LangChain documents
langchain_docs = artifact.to_langchain_documents()
print("LangChain document content:", langchain_docs[0].page_content)
print("LangChain document metadata:", langchain_docs[0].metadata)

# Add LangChain documents to collection
milvus_ids = collection.add_documents(langchain_docs)
print("Documents added with IDs:", milvus_ids)

# Search in collection
results = collection.similarity_search("sample", k=1)
print("Search results:", [doc.page_content for doc in results])

LangChain document content: Sample content for demonstration
LangChain document metadata: {'source': 'vectorstore-demo3/sample-doc', 'original_source': '/tmp/tmpawvdmdq5.txt', 'mlrun_object_uri': 'store://artifacts/vectorstore-demo3/sample-doc#0@eb00adb2de8042c4eae0d7b18b4a3797c2749dac^b9ab6e6eca457947c6a0cbfeeb456a71cd2e8798', 'mlrun_chunk': '0'}
Documents added with IDs: [454354693163582836]
Search results: ['Sample content for demonstration']


 ## Working with MLRun artifacts in a collection

In [123]:
# Add artifacts directly to collection
artifact1 = project.log_document(
    "doc1", local_path=create_sample_document("First document")
)
artifact2 = project.log_document(
    "doc2", local_path=create_sample_document("Second document")
)

# Add multiple artifacts at once
milvus_ids = collection.add_artifacts([artifact1, artifact2])
print("Artifacts added with IDs:", milvus_ids)

# Get back as LangChain documents
search_results = collection.similarity_search("first")
print("Retrieved document:", search_results[0].page_content)

Artifacts added with IDs: [454354693163582838, 454354693163582840]
Retrieved document: Sample content for demonstration


## Using text splitters with Artifacts

In [124]:
# Create a text splitter
splitter = CharacterTextSplitter(separator="\n", chunk_size=100, chunk_overlap=20)

# Create a longer document
long_text = "This is a longer document.\n" * 5
long_doc = project.log_document(
    "long-doc", local_path=create_sample_document(long_text)
)

# Add artifact with splitting
collection_split = project.get_vector_store_collection(
    vector_store=Milvus(
        collection_name="split_collection",
        embedding_function=embedding_model,
        connection_args=config["MILVUS_DB"],
        auto_id=False,
    ),
)

# Add with custom IDs for chunks
ids = collection_split.add_artifacts([long_doc], splitter=splitter, ids=["doc1"])
print("Generated chunk IDs:", ids)

Generated chunk IDs: ['doc1_1', 'doc1_2']


 Using MLRunLoader

In [125]:
# Create a document loader specification
loader_spec = DocumentLoaderSpec(
    loader_class_name="langchain_community.document_loaders.TextLoader",
    src_name="file_path",
)

# Create and use MLRunLoader
file_path = create_sample_document("Content for MLRunLoader test")
loader = MLRunLoader(
    source_path=file_path,
    loader_spec=loader_spec,
    artifact_key="loaded-doc",
    producer=project,
)

# Load documents
documents = loader.load()
print("Loaded document content:", documents[0].page_content)

# Verify artifact creation
artifact = project.get_artifact("loaded-doc")
print("Created artifact key:", artifact.key)

Loaded document content: Content for MLRunLoader test
Created artifact key: loaded-doc


Using MLRunLoader with DirectoryLoader

In [126]:
# Create a directory with multiple documents
temp_dir = tempfile.mkdtemp()
create_sample_document("First file content", dir=temp_dir)
create_sample_document("Second file content", dir=temp_dir)

# Configure loader specification
artifact_loader_spec = DocumentLoaderSpec(
    loader_class_name="langchain_community.document_loaders.TextLoader",
    src_name="file_path",
)

# Create directory loader with MLRunLoader
dir_loader = DirectoryLoader(
    temp_dir,
    glob="**/*.*",
    loader_cls=MLRunLoader,
    loader_kwargs={
        "loader_spec": artifact_loader_spec,
        "artifact_key": "dir_doc%%",  # %% will be replaced with unique identifier
        "producer": project,
        "upload": False,
    },
)

# Load all documents
documents = dir_loader.load()
print(f"Loaded {len(documents)} documents")

# List created artifacts
artifacts = project.list_artifacts(kind="document")
matching_artifacts = [
    art for art in artifacts if art["metadata"]["key"].startswith("dir_doc")
]

print("Created artifacts:", [art["metadata"]["key"] for art in matching_artifacts])

Loaded 2 documents
Created artifacts: ['dir_doc2ftmp2ftmppsy90gh72ftmp6y86mt4d.txt', 'dir_doc2ftmp2ftmppsy90gh72ftmp2w62cehp.txt']


Cleanup


In [127]:
# Retrieve all document-type artifacts from the project
doc_artifacts = project.list_artifacts(kind="document", tag="latest").to_objects()

# First pass: Remove collection references from all artifacts
for art in doc_artifacts:
    # Store the initial state of collections for comparison
    collections_before_remove = art.spec.collections.copy()

    # Remove this artifact's reference from the collection
    # Note: This only removes the reference, not the artifact itself
    if collections_before_remove:
        collection.remove_from_artifact(art)
        # Refresh artifact state after removal
        art = project.get_artifact(art.key)
        collections_after_remove = art.spec.collections.copy()
        # Print the before/after state of collections for this artifact
        print(f"{art.key}:{collections_before_remove}->{collections_after_remove}")

# Special handling for 'long-doc' artifact
# This artifact exists in a 'split_collection' that we don't have direct access to
art = project.get_artifact("long-doc")
collections_before_remove = art.spec.collections.copy()
# Remove reference to 'split_collection' from this artifact
art.collection_remove("split_collection")
project.update_artifact(art)
# Refresh and verify the changes
art = project.get_artifact("long-doc")
collections_after_remove = art.spec.collections.copy()
print(f"{art.key}:{collections_before_remove}->{collections_after_remove}")

# Final cleanup: Delete all document artifacts
doc_artifacts = project.list_artifacts(kind="document").to_objects()
for art in doc_artifacts:
    # Completely delete the artifact from the project
    project.delete_artifact(art)

# Finally, remove the underlying Milvus collection
collection.col.drop()

sample-doc:{'my_tutorial_collection': '1'}->{}
doc1:{'my_tutorial_collection': '1'}->{}
doc2:{'my_tutorial_collection': '1'}->{}
long-doc:{'split_collection': '1'}->{'split_collection': '1'}
long-doc:{'split_collection': '1'}->{}
